In [ ]:
!pip install -q kagglehub scikit-learn

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.nn.functional import cross_entropy
import numpy as np
import re, string
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from collections import Counter
import json
import kagglehub

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

VOCAB_SIZE = 10000
MAX_LEN = 80
EMBEDDING_DIM = 256
N_HEADS = 2
FF_DIM = 256
BATCH_SIZE = 32
EPOCHS = 10   # mets 100 si tu veux coller exactement au TP

Device: cuda


In [ ]:
path = kagglehub.dataset_download("zynicide/wine-reviews")
print(path)

Using Colab cache for faster access to the 'wine-reviews' dataset.
/kaggle/input/wine-reviews


In [ ]:
with open(path + "/winemag-data-130k-v2.json") as json_data:
    wine_data = json.load(json_data)

filtered_data = [
    "wine review : "
    + x["country"]
    + " : "
    + x["province"]
    + " : "
    + x["variety"]
    + " : "
    + x["description"]
    for x in wine_data
    if x["country"] is not None
    and x["province"] is not None
    and x["variety"] is not None
    and x["description"] is not None
]

print("Nombre d'exemples :", len(filtered_data))
print(filtered_data[0][:300])

Nombre d'exemples : 129907
wine review : Italy : Sicily & Sardinia : White Blend : Aromas include tropical fruit, broom, brimstone and dried herb. The palate isn't overly expressive, offering unripened apple, citrus and dried sage alongside brisk acidity.


In [ ]:
class SimpleTokenizer:
    def __init__(self, texts, vocab_size):
        self.vocab_size = vocab_size
        self.counter = Counter()

        for text in texts:
            self.counter.update(text.split())

        self.vocab = [word for word, _ in self.counter.most_common(vocab_size - 2)]
        self.word2idx = {word: idx + 2 for idx, word in enumerate(self.vocab)}
        self.word2idx["<pad>"] = 0
        self.word2idx["<unk>"] = 1

    def encode(self, text):
        return [self.word2idx.get(word, 1) for word in text.split()]

In [ ]:
def pad_punctuation(s):
    s = re.sub(f"([{string.punctuation}])", r" \1 ", s)
    s = re.sub(" +", " ", s)
    return s.lower()

tokenizer = SimpleTokenizer([pad_punctuation(t) for t in filtered_data], VOCAB_SIZE)
print("Vocab size effective:", len(tokenizer.word2idx))

Vocab size effective: 10000


In [ ]:
train_texts, test_texts = train_test_split(
    filtered_data,
    test_size=0.1,
    random_state=42
)

print("Train:", len(train_texts))
print("Test :", len(test_texts))

Train: 116916
Test : 12991


In [ ]:
class WineDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = self.tokenizer.encode(pad_punctuation(self.texts[idx]))[:self.max_len + 1]
        padding = [self.tokenizer.word2idx["<pad>"]] * (self.max_len + 1 - len(tokens))
        tokens += padding
        return torch.tensor(tokens[:-1], dtype=torch.long), torch.tensor(tokens[1:], dtype=torch.long)

In [ ]:
class GPTBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        seq_len = x.size(1)

        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
        mask = mask.masked_fill(mask == 0, float("-inf")).masked_fill(mask == 1, 0.0)

        attn_output, _ = self.attn(x, x, x, attn_mask=mask)
        x = self.ln1(x + attn_output)

        ffn_output = self.ffn(x)
        return self.ln2(x + ffn_output)

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, max_len, num_heads, ff_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embedding = nn.Embedding(max_len, embed_dim)
        self.transformer = GPTBlock(embed_dim, num_heads, ff_dim)
        self.fc = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        positions = torch.arange(0, x.size(1), device=x.device).unsqueeze(0)
        x = self.embedding(x) + self.pos_embedding(positions)
        x = self.transformer(x)
        return self.fc(x)

In [ ]:
model = GPTModel(VOCAB_SIZE, EMBEDDING_DIM, MAX_LEN, N_HEADS, FF_DIM).to(DEVICE)
optimizer = Adam(model.parameters(), lr=1e-4)

print(model)

GPTModel(
  (embedding): Embedding(10000, 256)
  (pos_embedding): Embedding(80, 256)
  (transformer): GPTBlock(
    (attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
    )
    (ffn): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=256, bias=True)
    )
    (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (fc): Linear(in_features=256, out_features=10000, bias=True)
)


In [ ]:
train_dataset = WineDataset(train_texts, tokenizer, MAX_LEN)
test_dataset = WineDataset(test_texts, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for x_batch, y_batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()

        logits = model(x_batch)
        loss = cross_entropy(
            logits.view(-1, VOCAB_SIZE),
            y_batch.view(-1),
            ignore_index=tokenizer.word2idx["<pad>"]
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}, Training Loss: {avg_loss:.4f}")

Epoch 1/10: 100%|██████████| 3654/3654 [01:10<00:00, 51.93it/s]


Epoch 1, Training Loss: 4.3488


Epoch 2/10: 100%|██████████| 3654/3654 [01:16<00:00, 47.73it/s]


Epoch 2, Training Loss: 3.6095


Epoch 3/10: 100%|██████████| 3654/3654 [01:14<00:00, 49.21it/s]


...

Epoch 8, Training Loss: 2.9394


Epoch 9/10: 100%|██████████| 3654/3654 [01:13<00:00, 49.40it/s]


Epoch 9, Training Loss: 2.8972


Epoch 10/10: 100%|██████████| 3654/3654 [01:13<00:00, 49.38it/s]

Epoch 10, Training Loss: 2.8597


In [ ]:
model.eval()
total_loss = 0.0

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        logits = model(x_batch)
        loss = cross_entropy(
            logits.view(-1, VOCAB_SIZE),
            y_batch.view(-1),
            ignore_index=tokenizer.word2idx["<pad>"]
        )

        total_loss += loss.item()

print("Test loss:", total_loss / len(test_loader))

Test loss: 2.9379189789588818


In [ ]:
def generate_text(model, tokenizer, start_prompt, max_tokens=80, temperature=1.0):
    model.eval()

    prompt = pad_punctuation(start_prompt)
    tokens = tokenizer.encode(prompt)
    tokens = tokens[:MAX_LEN]
    generated = tokens.copy()

    with torch.no_grad():
        for _ in range(max_tokens):
            input_tensor = torch.tensor([generated[-MAX_LEN:]], dtype=torch.long, device=DEVICE)
            logits = model(input_tensor)

            logits = logits[:, -1, :] / temperature
            probs = torch.softmax(logits, dim=-1)

            next_token = torch.multinomial(probs, num_samples=1).item()

            if next_token == tokenizer.word2idx["<pad>"]:
                break

            generated.append(next_token)

    idx2word = {idx: word for word, idx in tokenizer.word2idx.items()}
    generated_text = " ".join(idx2word.get(idx, "<unk>") for idx in generated)
    return generated_text

In [ ]:
prompt = "wine review : us"
generated_text = generate_text(model, tokenizer, prompt, max_tokens=50, temperature=0.8).strip()
print("Generated text:\n")
print(generated_text)

Generated text:

wine review : us : california : cabernet sauvignon : this is a dry cabernet , a robust , fine cabernet that ' s explosive for its personality . dry and tannic , it has a certain tannins more substantial , and the heat of its climate - class mouthfeel that will age well


In [ ]:
prompt = "wine review : france : bordeaux : merlot :"

for temp in [0.5, 0.8, 1.2]:
    print(f"\n--- Temperature = {temp} ---")
    print(generate_text(model, tokenizer, prompt, max_tokens=40, temperature=temp))


--- Temperature = 0.5 ---
wine review : france : bordeaux : merlot : this is a soft , full - bodied wine that has attractive red fruits and juicy acidity . it is still young , with a firm , fresh character and a rich texture . the wine is ready to drink

--- Temperature = 0.8 ---
wine review : france : bordeaux : merlot : this is a rich wine and dense , with ripe tannins . it is rich and richly textured with red - berry flavors . it ' s with great zing to give a wine that will be ready to drink

--- Temperature = 1.2 ---
wine review : france : bordeaux : merlot : dark gold plush in this wine happily delicately accessible malbec , even early was someone freedom hill ' s luxury quinta . tannins , concentration . it is the juicy and extracted , while does not tacky drinkability . coming


In [ ]:
torch.save(model.state_dict(), "gpt_wine_model.pth")
print("Modèle sauvegardé dans gpt_wine_model.pth")

Modèle sauvegardé dans gpt_wine_model.pth


In [ ]:
model.load_state_dict(torch.load("gpt_wine_model.pth", map_location=DEVICE))
model.eval()
print("Modèle rechargé")

Modèle rechargé
